# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Elvina
- **Email:** elvinasintang@gmail.com
- **ID Dicoding:** DS183B6X0511

## Menentukan Pertanyaan Bisnis

**SMART Question** adalah sebuah framework untuk merumuskan pertanyaan secara terstruktur agar memeperoleh informasi yang mendalam.

* **Spesific (Spesifik)**
  * Pertanyaan harus jelas, fokus pada sebuah topik tertentu, dan tidak bermakna ganda. Hindari pertanyaan yang terlalu luas.
    * Salah: Bagaimana cara meningkatkan penjualan?
    * Benar: Faktor apa saja yang memengaruhi penurunan penjualan produk kategori elektronik di wilayah Jakarta selama kuartal terakhir?
* **Measurable (Terukur)**
  * Pertanyaan harus bisa dijawab dengan angka atau matrix yang konkret, harus tahu apa yang akan dihitung.
    * Salah: Apakah pelanggan senang dengan layanan kita?
    * Benar: Berapa skor rata-rata Customer Satisfaction untuk layanan purna jual bulan ini dibandingkan bulan lalu?
* **Action-Oriented (Berorientasi Aksi)**
  * Hasil dari pertanyaan harus bisa memberikan arahan untuk melakukan tindakan nyata. Jika pertanyaan terjawab, stakeholder harus tahu apa langkah selanjutnya.
    * Salah: Mengapa orang suka berbelanja?
    * Benar: Fitur apa pada aplikasi yang paling sering digunakan sebelum pengguna memutuskan untuk melakukan checkout?
* **Relevant (Relevan)**
  * Hasil dari pertanyaan harus sejalan dengan tujuan utama bisnis atau masalah yang sedang dihadapi.
    * Salah: Menanyakan tentang stok gudang saat masalah utamanya adalah efektivitas kampanye media sosial.
    * Benar: Apakah kampanye iklan di Instagram memberikan Return on Ad Spend (ROAS) yang lebih tinggi dibandingkan iklan di TikTok?
* **Time-bound (Terikat Waktu)**
  * Pertanyaan harus ada batasan waktu yang jelas agar analisis memiliki konteks yang tepat.
    * Salah: Berapa banyak pengguna baru kita?
    * Benar: Berapa tingkat pertumbuhan pengguna baru secara bulanan (Month-over-Month) sepanjang tahun 2025?

**Contoh pertanyaan bisnis yang memenuhi seluruh elemen SMART**

***"Faktor apa saja yang memengaruhi penurunan conversion rate pada pengguna aplikasi Android di wilayah Jabodetabek sebesar 5% selama periode Flash Sale Maret 2026?"***

Keterangan:

- **Specific**: Fokus pada "penurunan conversion rate" untuk "aplikasi Android" di "Jabodetabek". Bukan sekadar penjualan turun.
- **Measurable**: Ada angka konkret yang ingin dianalisis, yaitu penurunan sebesar "5%".
- **Action-Oriented**: Dengan mengetahui faktor penyebabnya misalnya bug pada versi Android tertentu atau kendala logistik di Jabodetabek, tim bisa langsung melakukan perbaikan teknis atau operasional.
- **Relevant**: Penurunan konversi saat Flash Sale adalah masalah kritis bagi bisnis retail/e-commerce.
- **Time-bound**: Dibatasi pada periode spesifik "Flash Sale Maret 2026".

Konteks: sebuah penyedia layanan bike sharing ingin memahami pola penyewaan sepeda sepanjang tahun 2011-2012 untuk menyusun strategi ketersediaan armada dan promosi.

1. **Pertanyaan 1 (cuaca):** Bagaimana pengaruh kondisi cuaca (cerah, berawan, hujan/salju ringan) terhadap rata-rata jumlah penyewaan sepeda *harian* sepanjang 2011-2012, dan berapa persen selisihnya dibandingkan hari cerah?
2. **Pertanyaan 2 (pola jam):** Pada jam berapa puncak penyewaan sepeda terjadi pada hari kerja dibandingkan hari libur/akhir pekan sepanjang 2011-2012, dan berapa persentase pengguna *casual* vs *registered* pada masing-masing tipe hari?
3. **Pertanyaan 3 (pertumbuhan):** Berapa persentase pertumbuhan total penyewaan bulanan tahun 2012 dibandingkan 2011 (Year-over-Year), dan pada musim apa pertumbuhannya paling tinggi?

**Pemeriksaan SMART**
| Elemen | Pemenuhan |
|---|---|
| Specific | Masing-masing fokus pada satu topik: cuaca, jam puncak, pertumbuhan |
| Measurable | Rata-rata penyewaan, persentase selisih, jam puncak, persentase pengguna, persentase YoY |
| Action-Oriented | Hasilnya langsung dipakai untuk menentukan distribusi armada, jadwal perawatan, dan waktu promosi |
| Relevant | Sejalan dengan tujuan bisnis: memaksimalkan ketersediaan sepeda saat permintaan tinggi |
| Time-bound | Seluruh periode 2011-2012 |

## Import Semua Packages/Library yang Digunakan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## Data Wrangling

### Gathering Data

#### Load day_df dan hour_df

Dataset Bike Sharing terdiri dari dua berkas: `day.csv` (agregasi harian) dan `hour.csv` (agregasi per jam). `day.csv` dipakai untuk pertanyaan 1 dan 3, sedangkan `hour.csv` untuk pertanyaan 2.

> **Catatan Colab:** unggah `day.csv` dan `hour.csv` ke Colab (ikon folder di sisi kiri, lalu *Upload*) atau letakkan di folder `data/`. Kode di bawah membaca dari kedua lokasi tersebut.

In [ ]:
from pathlib import Path

# Membaca dari folder data/ (jika ada), jika tidak dari folder kerja saat ini
DATA_DIR = Path("data") if Path("data/day.csv").exists() else Path(".")

day_df = pd.read_csv(DATA_DIR / "day.csv")
hour_df = pd.read_csv(DATA_DIR / "hour.csv")

print("Ukuran day_df :", day_df.shape)
print("Ukuran hour_df:", hour_df.shape)
day_df.head()

In [ ]:
hour_df.head()

**Insight:**
- `day_df` berisi satu baris per hari, `hour_df` satu baris per jam, dengan kolom yang hampir sama (ditambah kolom `hr` pada data per jam).
- Kolom `temp`, `atemp`, `hum`, dan `windspeed` tampak berupa angka ternormalisasi (0-1), sehingga perlu dikembalikan ke satuan aslinya.
- Kolom `season`, `weathersit`, `weekday`, `yr`, `mnth` masih berupa kode angka.

### Assessing Data

#### Identifying missing value, duplikat, tipe data, invalid value, inconsistent value, dan outlier

In [ ]:
for name, df in [("day_df", day_df), ("hour_df", hour_df)]:
    print(f"===== {name} =====")
    df.info()
    print("Total missing value :", df.isna().sum().sum())
    print("Jumlah data duplikat:", df.duplicated().sum())
    print()

In [ ]:
day_df.describe()

In [ ]:
# 1. Tipe data: kolom tanggal masih bertipe object (string)
print("Tipe data dteday:", day_df["dteday"].dtype)

# 2. Nilai tidak valid / tidak akurat: kelembapan (hum) bernilai 0 itu tidak masuk akal
print("\nHari dengan hum == 0 pada day_df:")
display(day_df[day_df["hum"] == 0][["dteday", "hum", "windspeed", "cnt"]])
print("Jumlah jam dengan hum == 0 pada hour_df:", (hour_df["hum"] == 0).sum())

# 3. Kecepatan angin 0 (kemungkinan sensor tidak terbaca)
print("\nJam dengan windspeed == 0 pada hour_df:", (hour_df["windspeed"] == 0).sum(), "dari", len(hour_df), "baris")

# 4. Kelengkapan data per jam: seharusnya 731 hari x 24 jam
print("\nBaris hour_df:", len(hour_df), "| seharusnya:", 731 * 24)

# 5. Konsistensi: cnt harus sama dengan casual + registered
print("\nBaris cnt != casual + registered (day_df) :", (day_df["casual"] + day_df["registered"] != day_df["cnt"]).sum())
print("Baris cnt != casual + registered (hour_df):", (hour_df["casual"] + hour_df["registered"] != hour_df["cnt"]).sum())

# 6. Kategori cuaca yang muncul pada kedua dataset
print("\nKategori weathersit day_df :", sorted(day_df["weathersit"].unique()))
print("Kategori weathersit hour_df:", sorted(hour_df["weathersit"].unique()))

In [ ]:
# 7. Outlier dengan metode IQR (day_df)
def hitung_outlier(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum()

for kolom in ["cnt", "casual", "registered", "hum", "windspeed"]:
    print(f"{kolom:<11}: {hitung_outlier(day_df[kolom])} outlier")

**Steps to Take:**

- (1) ubah `dteday` ke datetime
- (2) beri label pada kolom kategori
- (3) kembalikan `temp`, `atemp`, `hum`, `windspeed` ke satuan asli
- (4) ganti `hum == 0` menjadi NaN lalu isi dengan median kelembapan pada bulan dan tahun yang sama
- (5) buang kolom yang tidak diperlukan.

**Insight (hasil assessing):**
- **Missing value dan duplikat:** tidak ditemukan pada kedua dataset.
- **Tipe data tidak sesuai:** `dteday` bertipe object, padahal seharusnya datetime.
- **Nilai tidak valid:** terdapat nilai kelembapan (`hum`) sama dengan 0, yang tidak masuk akal secara fisik. Kemungkinan sensor gagal membaca.
- **Nilai tidak akurat:** banyak nilai `windspeed` sama dengan 0 pada data per jam (kemungkinan sensor tidak mencatat). Kolom ini tidak dipakai untuk menjawab pertanyaan bisnis, sehingga hanya didokumentasikan.
- **Data per jam tidak lengkap:** jumlah baris `hour_df` kurang dari 731 x 24 karena ada jam tanpa penyewaan yang tidak tercatat. Ini bukan error, tetapi perlu diingat saat menghitung rata-rata per jam.
- **Data tidak konsisten secara representasi:** kolom kategori berupa kode angka dan kolom cuaca/suhu berupa angka ternormalisasi.
- **Outlier:** ada sejumlah outlier pada `casual`, `hum`, dan `windspeed`. Outlier pada penyewaan (`casual`) adalah lonjakan permintaan yang nyata, sehingga **dipertahankan**.

### Cleaning Data

#### Fixing tipe data, label kategori, satuan, dan nilai tidak valid

In [ ]:
season_map = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
weather_map = {1: "Cerah", 2: "Berawan", 3: "Hujan/Salju Ringan", 4: "Hujan/Salju Lebat"}
weekday_map = {0: "Minggu", 1: "Senin", 2: "Selasa", 3: "Rabu", 4: "Kamis", 5: "Jumat", 6: "Sabtu"}
year_map = {0: 2011, 1: 2012}


def bersihkan(df):
    df = df.copy()
    # 1. Tipe data tanggal
    df["dteday"] = pd.to_datetime(df["dteday"])
    # 2. Label kategori
    df["year"] = df["yr"].map(year_map)
    df["month"] = df["mnth"]
    df["season_label"] = df["season"].map(season_map)
    df["weather_label"] = df["weathersit"].map(weather_map)
    df["weekday_label"] = df["weekday"].map(weekday_map)
    df["day_type"] = np.where(df["workingday"] == 1, "Hari Kerja", "Hari Libur/Akhir Pekan")
    # 3. Kembalikan nilai ternormalisasi ke satuan asli
    df["temp_c"] = df["temp"] * 41
    df["atemp_c"] = df["atemp"] * 50
    df["hum_pct"] = df["hum"] * 100
    df["windspeed_kmh"] = df["windspeed"] * 67
    # 4. Buang kolom yang tidak dibutuhkan
    return df.drop(columns=["instant", "yr", "mnth", "temp", "atemp", "hum", "windspeed"])


day_df = bersihkan(day_df)
hour_df = bersihkan(hour_df)

# 5. Kelembapan 0 dianggap tidak valid -> NaN -> isi dengan median bulan-tahun yang sama
for df in (day_df, hour_df):
    df.loc[df["hum_pct"] == 0, "hum_pct"] = np.nan
    df["hum_pct"] = df["hum_pct"].fillna(df.groupby(["year", "month"])["hum_pct"].transform("median"))

print("hum_pct == 0 tersisa :", (day_df["hum_pct"] == 0).sum(), "|", (hour_df["hum_pct"] == 0).sum())
print("Missing value tersisa:", day_df.isna().sum().sum(), "|", hour_df.isna().sum().sum())
day_df.head()

**Insight:**
- Kolom tanggal sudah bertipe datetime, kolom kategori sudah berlabel, dan variabel cuaca sudah dalam satuan asli (suhu dalam °C, kelembapan dalam %, kecepatan angin dalam km/jam).
- Nilai kelembapan 0 sudah diganti dengan median bulan-tahun yang sama, sehingga tidak ada lagi nilai tidak valid.
- Kolom `windspeed_kmh` yang bernilai 0 dibiarkan karena tidak dipakai dalam analisis.

## Exploratory Data Analysis (EDA)

### Eksplorasi Pertanyaan 1: Pengaruh Cuaca terhadap Penyewaan Harian

In [ ]:
ringkasan_cuaca = (
    day_df.groupby("weather_label")["cnt"]
    .agg(jumlah_hari="count", rata_rata="mean", median="median", total="sum")
    .sort_values("rata_rata", ascending=False)
    .round(1)
)
dasar = ringkasan_cuaca.loc["Cerah", "rata_rata"]
ringkasan_cuaca["selisih_vs_cerah_persen"] = ((ringkasan_cuaca["rata_rata"] / dasar - 1) * 100).round(1)
ringkasan_cuaca

In [ ]:
# Pembanding: korelasi variabel cuaca numerik terhadap jumlah penyewaan harian
day_df[["cnt", "temp_c", "atemp_c", "hum_pct", "windspeed_kmh"]].corr()["cnt"].round(3)

### Eksplorasi Pertanyaan 2: Jam Puncak Hari Kerja vs Hari Libur

In [ ]:
pola_jam = (
    hour_df.groupby(["day_type", "hr"])["cnt"]
    .mean()
    .reset_index(name="rata_rata_penyewaan")
)

# Tiga jam tersibuk untuk setiap tipe hari
for tipe, grup in pola_jam.groupby("day_type"):
    print(f"--- {tipe} ---")
    print(grup.nlargest(3, "rata_rata_penyewaan").round(1).to_string(index=False))
    print()

In [ ]:
komposisi = hour_df.groupby("day_type")[["casual", "registered"]].sum()
komposisi["persen_casual"] = (komposisi["casual"] / (komposisi["casual"] + komposisi["registered"]) * 100).round(1)
komposisi["persen_registered"] = (100 - komposisi["persen_casual"]).round(1)
komposisi

### Eksplorasi Pertanyaan 3: Pertumbuhan Penyewaan 2012 vs 2011

In [ ]:
bulanan = day_df.groupby(["year", "month"])["cnt"].sum().unstack("year")
bulanan["yoy_persen"] = ((bulanan[2012] / bulanan[2011] - 1) * 100).round(1)
bulanan

In [ ]:
total_tahunan = day_df.groupby("year")["cnt"].sum()
print("Total penyewaan 2011:", total_tahunan[2011])
print("Total penyewaan 2012:", total_tahunan[2012])
print("Pertumbuhan YoY      :", round((total_tahunan[2012] / total_tahunan[2011] - 1) * 100, 1), "%")

musim = day_df.groupby(["season_label", "year"])["cnt"].sum().unstack("year")
musim = musim.reindex(["Spring", "Summer", "Fall", "Winter"])
musim["yoy_persen"] = ((musim[2012] / musim[2011] - 1) * 100).round(1)
musim

**Insight (EDA):**
- Cuaca memiliki pengaruh jelas: rata-rata penyewaan menurun dari cuaca cerah ke berawan, lalu ke hujan/salju ringan (angka pada tabel `ringkasan_cuaca`). Suhu berkorelasi positif dengan penyewaan, sedangkan kelembapan dan kecepatan angin berkorelasi negatif.
- Pola jam berbeda antara hari kerja dan hari libur: hari kerja memiliki dua puncak (jam berangkat dan pulang kerja), sedangkan hari libur memiliki satu puncak yang landai pada siang hingga sore.
- Pengguna *registered* mendominasi di semua tipe hari, tetapi porsi *casual* lebih besar pada hari libur/akhir pekan.
- Total penyewaan 2012 lebih tinggi daripada 2011 di seluruh bulan dan seluruh musim.

## Visualization & Explanatory Analysis

### Pertanyaan 1: Bagaimana pengaruh kondisi cuaca terhadap rata-rata penyewaan harian?

In [ ]:
data_plot = ringkasan_cuaca.reset_index()
warna = ["#1f77b4" if w == "Cerah" else "#c7c7c7" for w in data_plot["weather_label"]]

fig, ax = plt.subplots(figsize=(8, 5))
batang = ax.bar(data_plot["weather_label"], data_plot["rata_rata"], color=warna)

for rect, (_, baris) in zip(batang, data_plot.iterrows()):
    label = f"{baris['rata_rata']:,.0f}"
    if baris["weather_label"] != "Cerah":
        label += f"\n({baris['selisih_vs_cerah_persen']:+.1f}% vs cerah)"
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 60, label, ha="center", va="bottom", fontsize=10)

ax.set_title("Rata-rata Penyewaan Harian Menurun Seiring Memburuknya Cuaca", fontsize=13, fontweight="bold")
ax.set_xlabel("Kondisi Cuaca")
ax.set_ylabel("Rata-rata Penyewaan per Hari")
ax.set_ylim(0, data_plot["rata_rata"].max() * 1.25)
sns.despine()
plt.tight_layout()
plt.show()

**Penjelasan:** hari cerah memiliki rata-rata penyewaan tertinggi, lalu menurun pada hari berawan, dan turun tajam ketika hujan/salju ringan. Batang biru menandai kondisi acuan (cerah), sedangkan persentase pada batang lain menunjukkan selisihnya terhadap hari cerah.

### Pertanyaan 2: Kapan jam puncak hari kerja vs hari libur, dan bagaimana komposisi penggunanya?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})

# Panel kiri: pola per jam
warna_tipe = {"Hari Kerja": "#1f77b4", "Hari Libur/Akhir Pekan": "#ff7f0e"}
for tipe, grup in pola_jam.groupby("day_type"):
    axes[0].plot(grup["hr"], grup["rata_rata_penyewaan"], marker="o", linewidth=2, color=warna_tipe[tipe], label=tipe)
    puncak = grup.loc[grup["rata_rata_penyewaan"].idxmax()]
    axes[0].annotate(f"Puncak {int(puncak['hr']):02d}:00\n({puncak['rata_rata_penyewaan']:.0f})",
                     xy=(puncak["hr"], puncak["rata_rata_penyewaan"]),
                     xytext=(puncak["hr"] + 0.8, puncak["rata_rata_penyewaan"] + 20),
                     fontsize=9, color=warna_tipe[tipe])
axes[0].set_title("Pola Penyewaan per Jam: Hari Kerja vs Hari Libur", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Jam")
axes[0].set_ylabel("Rata-rata Penyewaan per Jam")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(title="Tipe Hari")

# Panel kanan: komposisi pengguna
komposisi[["persen_casual", "persen_registered"]].plot(
    kind="bar", stacked=True, ax=axes[1], color=["#ff7f0e", "#1f77b4"], width=0.6
)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f%%", label_type="center", color="white", fontsize=10)
axes[1].set_title("Komposisi Pengguna", fontsize=13, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Persentase Penyewaan (%)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(["Casual", "Registered"], loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)

sns.despine()
plt.tight_layout()
plt.show()

**Penjelasan:** pada hari kerja terdapat dua puncak tajam yang sejalan dengan jam berangkat dan pulang kerja, sedangkan pada hari libur kurva lebih landai dengan puncak pada siang hari. Pengguna registered mendominasi, tetapi porsi casual lebih besar pada hari libur.

### Pertanyaan 3: Bagaimana pertumbuhan penyewaan 2012 dibandingkan 2011?

In [ ]:
nama_bulan = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})

# Panel kiri: tren bulanan
axes[0].plot(nama_bulan, bulanan[2011], marker="o", linewidth=2, color="#c7c7c7", label="2011")
axes[0].plot(nama_bulan, bulanan[2012], marker="o", linewidth=2, color="#1f77b4", label="2012")
axes[0].fill_between(nama_bulan, bulanan[2011], bulanan[2012], color="#1f77b4", alpha=0.08)
axes[0].set_title("Total Penyewaan Bulanan: 2012 Konsisten Lebih Tinggi dari 2011", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Bulan")
axes[0].set_ylabel("Total Penyewaan")
axes[0].legend(title="Tahun")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Panel kanan: pertumbuhan YoY per musim
warna_musim = ["#1f77b4" if v == musim["yoy_persen"].max() else "#c7c7c7" for v in musim["yoy_persen"]]
batang = axes[1].bar(musim.index, musim["yoy_persen"], color=warna_musim)
axes[1].bar_label(batang, fmt="%.1f%%", padding=3)
axes[1].set_title("Pertumbuhan YoY per Musim", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Musim")
axes[1].set_ylabel("Pertumbuhan 2012 vs 2011 (%)")
axes[1].set_ylim(0, musim["yoy_persen"].max() * 1.2)

sns.despine()
plt.tight_layout()
plt.show()

**Penjelasan:** garis biru (2012) berada di atas garis abu-abu (2011) pada hampir seluruh bulan. Diagram batang menunjukkan musim dengan pertumbuhan tertinggi (disorot biru).

## Analisis Lanjutan (Opsional)

**Tujuan:** mengelompokkan data berdasarkan aturan bisnis tanpa algoritma machine learning, sehingga pola permintaan lebih mudah dibaca oleh pemangku kepentingan. Teknik RFM dan geospatial tidak dipakai karena dataset Bike Sharing tidak memiliki identitas pelanggan maupun koordinat lokasi, sehingga keduanya tidak relevan.

Tiga pengelompokan yang dibuat:
1. **Binning suhu** (`temp_category`): Dingin (< 15 °C), Sejuk (15-25 °C), Hangat (> 25 °C).
2. **Manual grouping waktu** (`time_of_day`): Dini Hari (00-05), Pagi (06-10), Siang (11-15), Sore (16-20), Malam (21-23).
3. **Binning tingkat permintaan** (`demand_cluster`): hari dibagi menjadi tiga kelompok sama besar berdasarkan kuantil `cnt` (Rendah, Sedang, Tinggi).

In [ ]:
urutan_suhu = ["Dingin (<15°C)", "Sejuk (15-25°C)", "Hangat (>25°C)"]
urutan_waktu = ["Dini Hari (00-05)", "Pagi (06-10)", "Siang (11-15)", "Sore (16-20)", "Malam (21-23)"]
urutan_permintaan = ["Rendah", "Sedang", "Tinggi"]

for df in (day_df, hour_df):
    df["temp_category"] = pd.cut(df["temp_c"], bins=[-np.inf, 15, 25, np.inf], labels=urutan_suhu)

hour_df["time_of_day"] = pd.cut(hour_df["hr"], bins=[-1, 5, 10, 15, 20, 23], labels=urutan_waktu)
day_df["demand_cluster"] = pd.qcut(day_df["cnt"], q=3, labels=urutan_permintaan)

print(day_df["demand_cluster"].value_counts().reindex(urutan_permintaan))

In [ ]:
# Proporsi tingkat permintaan pada setiap kategori suhu (dalam persen per baris)
silang_suhu = pd.crosstab(day_df["temp_category"], day_df["demand_cluster"], normalize="index").mul(100).round(1)
silang_suhu = silang_suhu.reindex(urutan_suhu)[urutan_permintaan]
silang_suhu

In [ ]:
rata_waktu = (
    hour_df.groupby(["time_of_day", "day_type"])["cnt"].mean().unstack("day_type").reindex(urutan_waktu).round(1)
)
rata_waktu

**Insight:** hari dengan suhu hangat didominasi oleh kelompok permintaan Tinggi, sedangkan hari dingin didominasi permintaan Rendah. Pengelompokan waktu memperlihatkan bahwa kelompok Pagi dan Sore menjadi kontributor terbesar pada hari kerja, sedangkan pada hari libur kelompok Siang dan Sore yang paling ramai.

### Visualisasi Analisis Lanjutan: Suhu, Tingkat Permintaan, dan Waktu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.heatmap(silang_suhu, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% hari"}, ax=axes[0])
axes[0].set_title("Sebaran Tingkat Permintaan pada Tiap Kategori Suhu (%)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Tingkat Permintaan Harian")
axes[0].set_ylabel("Kategori Suhu")

rata_waktu.plot(kind="bar", ax=axes[1], color=[warna_tipe[c] for c in rata_waktu.columns], width=0.75)
axes[1].set_title("Rata-rata Penyewaan per Jam menurut Waktu dalam Sehari", fontsize=13, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Rata-rata Penyewaan per Jam")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20)
axes[1].legend(title="Tipe Hari")

sns.despine()
plt.tight_layout()
plt.show()

## Conclusion & Recommendation

Sel berikut menghitung ulang angka-angka utama agar kesimpulan di bawahnya konsisten dengan hasil analisis.

In [ ]:
selisih = ringkasan_cuaca["selisih_vs_cerah_persen"]
print("[Pertanyaan 1]")
for cuaca, nilai in selisih.items():
    if cuaca != "Cerah":
        print(f"  Cuaca '{cuaca}': rata-rata {ringkasan_cuaca.loc[cuaca, 'rata_rata']:,.0f} penyewaan/hari ({nilai:+.1f}% dibanding cerah)")
print(f"  Cuaca 'Cerah': rata-rata {dasar:,.0f} penyewaan/hari")

print("\n[Pertanyaan 2]")
for tipe, grup in pola_jam.groupby("day_type"):
    puncak = grup.nlargest(3, "rata_rata_penyewaan")["hr"].sort_values().tolist()
    print(f"  {tipe}: tiga jam tersibuk = {[f'{h:02d}:00' for h in puncak]}")
for tipe, baris in komposisi.iterrows():
    print(f"  {tipe}: casual {baris['persen_casual']}% | registered {baris['persen_registered']}%")

print("\n[Pertanyaan 3]")
print(f"  Pertumbuhan total 2012 vs 2011: {(total_tahunan[2012] / total_tahunan[2011] - 1) * 100:.1f}%")
musim_terbaik = musim["yoy_persen"].idxmax()
print(f"  Musim dengan pertumbuhan tertinggi: {musim_terbaik} ({musim.loc[musim_terbaik, 'yoy_persen']:.1f}%)")

- **Conclusion pertanyaan 1 (Pengaruh Cuaca):**
Cuaca berpengaruh kuat terhadap penyewaan harian. Rata-rata penyewaan tertinggi terjadi pada hari cerah, turun pada hari berawan, dan turun paling tajam pada hari hujan/salju ringan (angka persentase lihat output sel di atas). Artinya, permintaan sangat sensitif terhadap kondisi cuaca.


- **Conclusion pertanyaan 2 (Jam Puncak dan Tipe Pengguna):**
Pada hari kerja permintaan memuncak dua kali, yaitu pada jam berangkat dan pulang kerja, sehingga sepeda dipakai terutama untuk komuting oleh pengguna registered. Pada hari libur permintaan memuncak pada siang hingga sore dengan pola lebih landai, dan porsi pengguna casual lebih besar dibandingkan hari kerja.


- **Conclusion pertanyaan 3 (Pertumbuhan 2012 vs 2011):**
Penyewaan tumbuh signifikan pada 2012 dibandingkan 2011 di seluruh bulan dan musim. Pertumbuhan tertinggi berada pada musim yang tercantum pada output di atas, sedangkan musim lainnya tumbuh lebih rendah tetapi tetap positif.


- **Conclusion analisis lanjutan:**
Hari bersuhu hangat cenderung masuk kelompok permintaan Tinggi, sedangkan hari dingin cenderung masuk kelompok Rendah. Suhu merupakan faktor lingkungan yang paling berkaitan dengan permintaan.

**Rekomendasi Action Item:**
1. **Penjadwalan armada berbasis jam:** pastikan ketersediaan dan redistribusi sepeda di stasiun area perkantoran/permukiman sebelum puncak pagi dan sore pada hari kerja, dan geser fokus ke area wisata/taman pada siang hari saat akhir pekan.
2. **Antisipasi cuaca:** gunakan prakiraan cuaca untuk menyesuaikan jumlah sepeda yang disebar dan menjadwalkan perawatan pada hari yang diperkirakan hujan, karena permintaan turun tajam.
3. **Promosi pengguna casual:** tawarkan paket akhir pekan atau promo keanggotaan kepada pengguna casual pada hari libur untuk mengonversi mereka menjadi pengguna registered.
4. **Perencanaan kapasitas musiman:** siapkan penambahan armada menjelang musim dengan pertumbuhan dan suhu tertinggi, dan manfaatkan musim dengan permintaan rendah untuk perawatan besar.

## Menyimpan Data Bersih untuk Dashboard

In [ ]:
import os

os.makedirs("dashboard", exist_ok=True)
day_df.to_csv("dashboard/main_data.csv", index=False)
hour_df.to_csv("dashboard/hour_data.csv", index=False)
print("Data bersih tersimpan di folder dashboard/ (main_data.csv dan hour_data.csv)")